In [1]:
import re 

# Read main.tex file from data/arxiv directory
with open('../data/arxiv/DPO.tex', 'r') as f:
    main_tex = f.read()

In [2]:

def clean_latex(latex_content: str) -> str:
    """
    Cleans a LaTeX string by extracting title, abstract, and main body,
    and removing commands and environments that don't contribute to the main text.
    """
    # 1. Extract title
    title_match = re.search(r'\\title\{(.*?)\}', latex_content, re.DOTALL)
    title = ""
    if title_match:
        title = title_match.group(1)
        title = re.sub(r'\\\\\s*', ' ', title)
        title = title.strip()

    # 2. Extract abstract
    abstract_match = re.search(r'\\begin\{abstract\}(.*?)\\end\{abstract\}', latex_content, re.DOTALL)
    abstract = abstract_match.group(1).strip() if abstract_match else ""

    # 3. Find start of main body (first section after document environment)
    body_start_index = -1
    doc_start_match = re.search(r'\\begin\{document\}', latex_content)
    if doc_start_match:
        # Find the first section after \begin{document}
        section_match = re.search(r'\\section', latex_content[doc_start_match.end():])
        if section_match:
            body_start_index = doc_start_match.end() + section_match.start()

    if body_start_index == -1:
        # Fallback if structure is unexpected, return what we have so far
        body = ""
    else:
        body = latex_content[body_start_index:]
    
    # 4. Find end of main body (before references, appendix, etc.)
    end_markers = [
        r'\\begin\{thebibliography\}', r'\\bibliography', r'\\appendix',
        r'\\section\*?\{Acknowledgements\}', r'\\section\*?\{Author Contributions\}'
    ]
    end_index = len(body)
    for marker_regex in end_markers:
        end_match = re.search(marker_regex, body)
        if end_match:
            end_index = min(end_index, end_match.start())
    
    body = body[:end_index]

    # Combine the parts we want to keep
    full_text = f'\\title{{{title}}}\n\n\\begin{{abstract}}\n{abstract}\n\\end{{abstract}}\n\n{body}\n\\end{{document}}'

    # Remove excessive new lines 
    full_text = re.sub(r'\n{3,}', '\n\n', full_text)
    
    # Now apply cleaning operations from the original function
    cleaned_text = full_text
    
    # # Environments to remove completely with their content
    # envs_to_remove = [
    #     'figure', 'figure\*', 'table', 'table\*', 'tabular', 'tabular\*', 'algorithm2e',
    #     'equation', 'equation\*', 'align', 'align\*', 'multline', 'multline\*',
    #     'wrapfigure', 'wraptable'
    # ]
    # for env in envs_to_remove:
    #     cleaned_text = re.sub(r'\\begin{' + env + r'}.*?\\end{' + env + r'}', '', cleaned_text, flags=re.DOTALL)

    # # Replace commands that have content we want to keep
    # cmds_keep_content = [
    #     'section', 'subsection', 'subsubsection', 'paragraph', 'subparagraph',
    #     'textbf', 'textit', 'emph', 'texttt', 'caption'
    # ]
    # for cmd in cmds_keep_content:
    #     cleaned_text = re.sub(r'\\' + cmd + r'\{([^}]+)\}', r'\1', cleaned_text)

    # # Handle \rev{old text}{new text} -> new text
    # cleaned_text = re.sub(r'\\rev\{[^}]*\}\{([^}]+)\}', r'\1', cleaned_text)

    # # Remove commands with arguments that we want to discard
    # cmds_remove_arg = ['label', 'ref', 'cite', 'citep', 'input', 'url']
    # for cmd in cmds_remove_arg:
    #     cleaned_text = re.sub(r'\\' + cmd + r'\{[^}]*\}', '', cleaned_text)

    # # Remove commands that don't have arguments
    # cmds_to_remove = [
    #     'maketitle', 'clearpage', 'AND', 'And', 'footnotemark', 'thanks', 'appendix', 'newpage'
    # ]
    # for cmd in cmds_to_remove:
    #     cleaned_text = re.sub(r'\\' + cmd + r'(?!\w)', '', cleaned_text)
    
    # # Remove custom commands from this specific paper
    # custom_cmds_to_remove = ['piref', 'pisft', 'methodac', 'methodfull', 'se']
    # for cmd in custom_cmds_to_remove:
    #     if '{' in cmd:
    #          cleaned_text = re.sub(r'\\' + cmd.split('{')[0] + r'\{[^}]*\}', '', cleaned_text)
    #     else:
    #         cleaned_text = re.sub(r'\\' + cmd + r'(?!\w)', '', cleaned_text)


    # # Remove environment tags but keep content
    # envs_keep_content = ['sproof'] # abstract and document are handled
    # for env in envs_keep_content:
    #     cleaned_text = re.sub(r'\\begin{' + env + r'\}', '', cleaned_text)
    #     cleaned_text = re.sub(r'\\end{' + env + r'\}', '', cleaned_text)

    # # Handle lists
    # cleaned_text = re.sub(r'\\begin{itemize}', '', cleaned_text)
    # cleaned_text = re.sub(r'\\end{itemize}', '', cleaned_text)
    # cleaned_text = re.sub(r'\\begin{enumerate}', '', cleaned_text)
    # cleaned_text = re.sub(r'\\end{enumerate}', '', cleaned_text)
    # cleaned_text = re.sub(r'\\item', '\n- ', cleaned_text)

    # # Remove comments
    # cleaned_text = re.sub(r'%.*', '', cleaned_text)
    
    # # Remove inline math expressions
    # cleaned_text = re.sub(r'\$.*?\$', '', cleaned_text)
    
    # # Clean up whitespace
    # cleaned_text = re.sub(r'~', ' ', cleaned_text)
    # cleaned_text = re.sub(r'\\ ', ' ', cleaned_text) # Explicit space command
    # cleaned_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', cleaned_text)
    # cleaned_text = re.sub(r'\n\s*\n', '\n\n', cleaned_text)  # Collapse multiple newlines
    # cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)  # Collapse multiple spaces
    # cleaned_text = cleaned_text.strip()

    return cleaned_text.strip()

In [3]:
cleaned_text = clean_latex(main_tex)
print(cleaned_text)

# Save the cleaned text
with open('../data/arxiv/cleaned_DPO.txt', 'w', encoding='utf-8') as f:
    f.write(cleaned_text)

\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract}
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.
\rev{In this paper, we leverage a mapping between reward functions and optimal policies to show that this constrained reward maximization proble

Let's paraphrase the article 10 times

In [23]:
# add .. path 
import sys
sys.path.append('..')
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

import utils.utils as utils

def paraphrase_paragraph(part):
    if len(part) < 200:
        return part
    else:
        return utils.query_llm(
            prompt=f"Paraphrase the following text, while keeping the latex tags. Try to paraphrase the text as much as possible. Text: {part}\n\nParaphrased text: ",
            model="gpt-4.1-mini", 
            temperature=1,
            top_p=0.9,
            system_prompt_included=False,
        )

def paraphrase_text(text):
    paragraphs = text.split('\n\n')
    # Parallelize the API calls
    with ThreadPoolExecutor(max_workers=8) as executor:
        paraphrased_paragraphs = list(tqdm(
            executor.map(paraphrase_paragraph, paragraphs),
            total=len(paragraphs)
        ))
    return '\n\n'.join(paraphrased_paragraphs)

In [ ]:
for i in range(10):
    with open(f'../data/arxiv/cleaned_DPO_paraphrased_{i}.txt', 'w') as f:
        paraphrased_text = paraphrase_text(cleaned_text)
        f.write(paraphrased_text)

100%|██████████| 45/45 [00:31<00:00,  1.42it/s]


\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract}
Although large-scale unsupervised language models (LMs) acquire extensive world knowledge and certain reasoning abilities, precisely steering their outputs remains challenging due to the fully unsupervised nature of their training process. Current approaches to achieve such steerability rely on gathering human judgments on the relative quality of model outputs and subsequently fine-tuning the unsupervised LM to align with these preferences, often using reinforcement learning from human feedback (RLHF). Nonetheless, RLHF involves a complicated and sometimes unstable pipeline: first training a reward model that captures human preferences, and then fine-tuning the large LM via reinforcement learning to optimize this learned reward while preventing significant deviation from the original model. \rev{In this paper, we leverage a mapping between reward functions and optimal policies to sh

100%|██████████| 45/45 [00:31<00:00,  1.43it/s]


\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract}
Although large-scale unsupervised language models (LMs) acquire extensive general knowledge and some reasoning capabilities, precisely steering their behavior remains challenging due to the fully unsupervised nature of their training. Current approaches to achieve such controllability involve gathering human judgments on the relative quality of model outputs and then fine-tuning the unsupervised LM to align with these preferences, typically via reinforcement learning from human feedback (RLHF). Nevertheless, RLHF is a complicated and often unstable process, requiring first the training of a reward model that captures human preferences, followed by reinforcement learning to adjust the large unsupervised LM in order to maximize this estimated reward without deviating excessively from the original model. \rev{In this paper, we leverage a mapping between reward functions and optimal pol

 22%|██▏       | 10/45 [00:14<00:47,  1.36s/it]

In [22]:
# Exploration; making sure the function works

part = cleaned_text.split('\n\n')[2]
utils.query_llm(
            prompt=f"Paraphrase the following text, while keeping the latex tags. Try to paraphrase the text as much as possible. Text: {part}\n\nParaphrased text: ",
            model="gpt-4.1-mini", 
            temperature=1,
            top_p=0.9,
            system_prompt_included=False,
        )

'\\section{Introduction}  \nLarge-scale unsupervised language models (LMs) trained on vast corpora develop remarkable abilities~\\citep{chowdhery2022palm, brown2020language, touvron2023llama,bubeck2023sparks}. Nonetheless, these models learn from human-generated data, which reflects a diverse set of intentions, priorities, and expertise. Not all of these intentions and skills are desirable to replicate; for instance, while we want an AI coding assistant to \\textit{recognize} typical programming errors to fix them, we also prefer it to favor producing the (sometimes infrequent) high-quality code examples found in its training set. Similarly, a language model might need to be \\textit{informed} about a misconception held by roughly half of the population, but it should not assert this falsehood as correct in 50\\% of its responses. Put differently, carefully choosing the model’s \\emph{preferred outputs and conduct} from its broad \\textit{knowledge and capabilities} is vital for creati

let's paraphrase as much as we can

In [ ]:
# Exploration; making sure the function works

part = cleaned_text.split('\n\n')[2]
utils.query_llm(
            prompt=f"Read the following text and paraphrase it. Make sure that the syntax, sentence structure, and diction is completely different. For instance, if the original paragraph is 'Sally went to the store and bought a new dress because she was sad.', the paraphrased text should be 'Sally was sad. She went to the store and bought a new dress.'. Also keep the latex formatting. Text: {part}\n\nParaphrased text: ",
            model="gpt-4.1-mini", 
            temperature=1,
            top_p=0.9,
            system_prompt_included=False,
        )

'\\section{Introduction}  \nUnsupervised language models (LMs) of substantial size, trained on extensive datasets, develop unexpectedly powerful abilities~\\citep{chowdhery2022palm, brown2020language, touvron2023llama,bubeck2023sparks}. Nevertheless, their training data originates from human sources with diverse intentions, priorities, and expertise levels. Some of these human traits may be undesirable to replicate; for instance, although an AI coding assistant should \\textit{recognize} typical programming errors to fix them, it is preferable for the model to favor the (often uncommon) examples of high-quality code generation it encountered during training. Likewise, a language model might need to be \\textit{cognizant} of widespread misconceptions held by roughly half the population, but it should never assert those falsehoods as truths in half of the instances it responds to regarding them. Put differently, carefully choosing which \\emph{responses and behaviors} a model exhibits fr